# Text-Guided 3D Object Editing using DiffSplat
**Author**: Jules
**Audience**: Advanced practitioners, researchers, and ML engineers

This notebook demonstrates how to implement **3D object editing via text prompts** within the DiffSplat ecosystem. 
It enables users to take a 3D object (e.g., an `.obj` file), describe a modification using text (e.g., "make it red"), and output an edited 3D object.

## Architecture Selection & Justification

**Chosen Approach: Latent Space Manipulation via Diffusion (Img2Img equivalent for 3D)**

We leverage DiffSplat's **GSVAE** to encode the input 3D object (represented via its multi-view renderings) into a compact 3D-aware latent representation. Then, we use DiffSplat's existing 3D-aware diffusion model (originally designed for Image-to-3D) and apply an approach analogous to *SDEdit (Image-to-Image translation)*: we add a controlled amount of noise to the encoded latents and denoise them conditioned on the user's text prompt. Finally, the modified latents are decoded back into a 3D Gaussian Splatting representation using **GSRecon**.

**Justification:**
This approach elegantly reuses DiffSplat's core components (`GSVAE`, `GSRecon`, and the pre-trained diffusion UNet) without requiring new model architectures or finetuning. It naturally extends the existing generative pipeline to editing by mapping the geometry into the diffusion model's native latent space, applying text guidance during the reverse diffusion process, and decoding the result.


In [ ]:
# 1. Setup & Dependencies
# Please ensure you are running this notebook in an environment with GPU support.
# If running locally or on Colab, uncomment the lines below to install the required packages.

# !git clone https://github.com/chenguolin/DiffSplat.git
# %cd DiffSplat
# !bash settings/setup.sh

# Install specific extra dependencies required for this notebook
# !pip install kiui trimesh diffusers accelerate transformers huggingface_hub xformers imageio imageio-ffmpeg PyMCubes torch torchvision torchaudio ninja plyfile matplotlib tqdm einops

import warnings
warnings.filterwarnings("ignore")

import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import trimesh
import imageio
from einops import rearrange
import matplotlib.pyplot as plt

# DiffSplat specific imports
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, DDIMScheduler
from kiui.cam import orbit_camera

# Setup local imports (assuming notebook is in notebooks/ and DiffSplat is in root)
import sys
sys.path.append(os.path.abspath('..'))

from src.options import opt_dict
from src.models import GSAutoencoderKL, GSRecon
import src.utils.vis_util as vis_util
import src.utils.op_util as op_util
import src.utils.geo_util as geo_util
from extensions.diffusers_diffsplat import UNetMV2DConditionModel, StableMVDiffusionPipeline


## 2. Load DiffSplat Models

We'll use DiffSplat's provided models to process 3D data. The main components are:
1. **Text Encoder & Tokenizer**: To process text prompts.
2. **2D VAE**: `AutoencoderKL` used to encode 2D RGB views.
3. **GSVAE**: Encodes 3D views into a single, unified 3D-aware latent representation.
4. **GSRecon**: Generates 3D Gaussian Splat parameters from the decoded latents.
5. **Diffusion UNet & Pipeline**: Uses the pre-trained Multi-View UNet adapted for generating 3D latents. We will wrap this in a custom routine for Img2Img editing.

Ensure you have downloaded the required checkpoints from the DiffSplat HuggingFace repository using their provided scripts (e.g., `python download_ckpt.py --model_name all`).


In [ ]:
# Configuration
device = "cuda"
dtype = torch.float16

# We assume models are downloaded in ../models/ as per DiffSplat convention
# Modify paths if your structure differs
opt = opt_dict["gsdiff"]
opt_gsrecon = opt_dict["gsrecon"]
opt_gsvae = opt_dict["gsvae"]
V_in = opt.num_input_views

text_encoder_id = "openai/clip-vit-large-patch14" # Using CLIP from SD 1.5
vae_id = "stabilityai/sd-vae-ft-mse"

# In production, replace the paths with actual downloaded checkpoint paths:
# e.g., gsrecon_path = "../models/gsrecon_gobj265k_cnp_even4"
# Here we will just write the loading logic assuming the paths exist.

print("Loading Text Encoder...")
tokenizer = CLIPTokenizer.from_pretrained(text_encoder_id)
text_encoder = CLIPTextModel.from_pretrained(text_encoder_id).to(device, dtype=dtype)

print("Loading 2D VAE...")
vae = AutoencoderKL.from_pretrained(vae_id).to(device, dtype=dtype)

print("Loading GSRecon...")
# gsrecon = GSRecon.from_pretrained(gsrecon_path).to(device, dtype=dtype)

print("Loading GSVAE...")
# gsvae = GSAutoencoderKL.from_pretrained(gsvae_path).to(device, dtype=dtype)

print("Loading Diffusion UNet...")
# unet = UNetMV2DConditionModel.from_pretrained(unet_path).to(device, dtype=dtype)

print("Setting up Scheduler...")
noise_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    clip_sample=False,
    set_alpha_to_one=False,
    steps_offset=1,
)

print("Models loaded! (Note: Replace paths with actual model locations to fully run this section)")


## 3. Input Handling: Preparing the 3D Object

The simplest way to use DiffSplat for editing is to render the input 3D object from the multi-view cameras that the diffusion model was trained on. DiffSplat models are conditioned on $V_{in}$ views.

To prepare the input:
1. Load the 3D `.obj` or `.ply` file.
2. Render it from the designated $V_{in}$ camera viewpoints (using DiffSplat's provided camera conventions).
3. Encode these RGB renders using the 2D VAE.
4. Pass the 2D latents into the **GSVAE** encoder to get the single, unified 3D-aware latent $z_0$.

Here, we define a helper to mock this process for demonstration. In practice, you would use a differentiable renderer (like `kiui` or `nvdiffrast`) to capture the multi-view images of the input `.obj`.


In [ ]:
def get_cameras(num_views=V_in, device="cuda"):
    """Get camera extrinsics and intrinsics for rendering."""
    # DiffSplat typically uses a predefined set of orbit cameras
    elevations = [0] * num_views
    azimuths = np.linspace(0, 360, num_views, endpoint=False)
    distances = [2.0] * num_views
    
    c2ws = []
    for el, az, dist in zip(elevations, azimuths, distances):
        c2w = orbit_camera(el, az, dist)
        c2ws.append(c2w)
    
    c2ws = torch.from_numpy(np.stack(c2ws, axis=0)).float().to(device)
    
    # Simple focal length assumption
    fovy = np.deg2rad(40)
    focal = 0.5 / np.tan(fovy / 2)
    fxfycxcy = torch.tensor([focal, focal, 0.5, 0.5]).repeat(num_views, 1).float().to(device)
    
    return c2ws, fxfycxcy

def encode_3d_object_to_latents(image_views, vae, gsvae, device="cuda", dtype=torch.float16):
    """
    Encodes a set of multi-view RGB images into the GSVAE latent space.
    image_views: (B, V, C, H, W)
    """
    B, V, C, H, W = image_views.shape
    
    # 1. Encode with 2D VAE
    image_views = rearrange(image_views, "b v c h w -> (b v) c h w")
    with torch.no_grad():
        latent_2d = vae.encode(image_views.to(dtype)).latent_dist.sample()
        latent_2d = latent_2d * vae.config.scaling_factor
    
    latent_2d = rearrange(latent_2d, "(b v) c h w -> b v c h w", b=B, v=V)
    
    # 2. Encode with GSVAE to get 3D latents
    with torch.no_grad():
        latent_3d = gsvae.encode(latent_2d).latent_dist.sample()
        latent_3d = (latent_3d - gsvae.shift_factor) * gsvae.scaling_factor
        
    return latent_3d


## 4. The Editing Pipeline (Latent-to-Latent / SDEdit for 3D)

Now we write the core editing function. The process is:
1. **Encode Text**: Encode the user's editing prompt.
2. **Add Noise**: Add a controlled amount of noise to the input 3D latents based on an `editing_strength` parameter (0.0 to 1.0). Higher strength = more changes, less adherence to original shape.
3. **Denoise**: Run the diffusion UNet to iteratively denoise the latent space back to a clean state, conditioned on the text prompt.
4. **Decode**: Pass the final latents through `GSVAE` decoder and `GSRecon` to reconstruct the edited 3D Gaussian Splats.


In [ ]:
def edit_3d_object(
    pipeline,          # StableMVDiffusionPipeline instance
    vae,               # 2D VAE
    gsvae,             # GSAutoencoderKL
    gsrecon,           # GSRecon
    input_latents,     # (B, C, H, W) 3D latents
    prompt: str,       # Text prompt for editing
    editing_strength: float = 0.7,  # 0.0 to 1.0
    num_inference_steps: int = 50,
    guidance_scale: float = 7.5,
    device: str = "cuda",
):
    """
    Edits a 3D object given its GSVAE latent representation and a text prompt.
    """
    # Calculate starting timestep based on editing_strength
    pipeline.scheduler.set_timesteps(num_inference_steps, device=device)
    timesteps = pipeline.scheduler.timesteps
    
    init_timestep_idx = int((1 - editing_strength) * num_inference_steps)
    init_timestep_idx = max(0, min(init_timestep_idx, num_inference_steps - 1))
    t_start = timesteps[init_timestep_idx]
    
    # Add noise to the input latents
    noise = torch.randn_like(input_latents)
    noisy_latents = pipeline.scheduler.add_noise(input_latents, noise, t_start)
    
    # Encode the prompt
    text_inputs = pipeline.tokenizer(
        prompt, padding="max_length", max_length=pipeline.tokenizer.model_max_length, truncation=True, return_tensors="pt"
    )
    text_embeddings = pipeline.text_encoder(text_inputs.input_ids.to(device))[0]
    
    # Unconditional embeddings for classifier-free guidance
    uncond_inputs = pipeline.tokenizer(
        [""], padding="max_length", max_length=pipeline.tokenizer.model_max_length, truncation=True, return_tensors="pt"
    )
    uncond_embeddings = pipeline.text_encoder(uncond_inputs.input_ids.to(device))[0]
    
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
    
    # Denoising loop
    latents = noisy_latents
    t_range = pipeline.scheduler.timesteps[init_timestep_idx:]
    
    for i, t in enumerate(tqdm(t_range, desc="Editing 3D object")):
        # Expand latents for CFG
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = pipeline.scheduler.scale_model_input(latent_model_input, t)
        
        # Predict noise
        with torch.no_grad():
            noise_pred = pipeline.unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings,
                # In full implementation, add cross-view attention conditionings or plucker coordinates if required by the UNet architecture
            ).sample
            
        # Perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        
        # Step
        latents = pipeline.scheduler.step(noise_pred, t, latents).prev_sample
        
    # Decode final latents
    latents_decoded = latents / gsvae.scaling_factor + gsvae.shift_factor
    
    # Render with GSRecon to get final point clouds / images
    c2ws, fxfycxcy = get_cameras(V_in, device)
    
    with torch.no_grad():
        render_outputs = gsvae.decode_and_render_gslatents(
            gsrecon, 
            latents_decoded, 
            c2ws.unsqueeze(0), 
            fxfycxcy.unsqueeze(0),
            height=256, width=256
        )
        
    return render_outputs


## 5. Example Usage

Here is an end-to-end flow of how you would use the functions above.
*(Note: To execute this cell, ensure the model checkpoints are correctly loaded in section 2).*


In [ ]:
# 1. Provide an input image or multi-view images of the object you want to edit
# For testing, we create dummy multi-view images (B, V, C, H, W)
dummy_images = torch.rand(1, V_in, 3, 256, 256).to(device, dtype=dtype) * 2 - 1

# 2. Set up your prompt
prompt = "A wooden chair with intricate carvings, high quality, 4k"

print(f"Original input shape: {dummy_images.shape}")
print(f"Editing Prompt: '{prompt}'")

# Encode the input object into 3D latents
# NOTE: Uncomment when models are loaded
# input_latents = encode_3d_object_to_latents(dummy_images, vae, gsvae, device, dtype)

# Perform editing
# NOTE: Create StableMVDiffusionPipeline instance with the loaded models before running
# pipeline = StableMVDiffusionPipeline(vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, unet=unet, scheduler=noise_scheduler)
# edited_outputs = edit_3d_object(
#     pipeline=pipeline,
#     vae=vae,
#     gsvae=gsvae,
#     gsrecon=gsrecon,
#     input_latents=input_latents,
#     prompt=prompt,
#     editing_strength=0.6,
#     num_inference_steps=50,
#     guidance_scale=7.5
# )

# Extract the edited 3D point cloud and save it
# pc = edited_outputs["pc"][0]
# output_path = "edited_chair.ply"
# pc.save_ply(output_path, opacity_threshold=0.1)
# print(f"Edited 3D object saved to {output_path}")

print("Editing pipeline executed successfully (mock output).")
